In [2]:
import openai
import nltk
import re, os
from dotenv import load_dotenv
import os
from openai import OpenAI

load_dotenv()

client = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY"),  # This is the default and can be omitted
)

# chat_completion = client.chat.completions.create(
#     messages=[
#         {
#             "role": "user",
#             "content": "Say this is a test",
#         }
#     ],
#     model="gpt-3.5-turbo",
#     # model="gpt-4o",
# )

In [6]:
# Sample ESG text
esg_text = """
Our company emitted 2000 tons of CO2 in 2023, which reflects a significant reduction in carbon emissions.
The diversity ratio of our workforce is now 45%, showing improvement in inclusivity.
We have achieved 70% renewable energy usage this year, a notable increase from last year.
Currently, women occupy 35% of leadership roles in our company, surpassing prior goals.
Our profits increased by 10% compared to last year, which is unrelated to our ESG efforts.
We are committed to sustainability and aim to reduce waste by 20%.
"""

# Define ESG metrics keywords
esg_keywords = ["carbon emissions", "diversity ratio", "renewable energy usage", "women in leadership"]

# Function to filter sentences related to ESG metrics
def filter_esg_sentences(text, keywords):
    sentences = nltk.sent_tokenize(text)  # Tokenize the text into sentences
    filtered_sentences = []

    for sentence in sentences:
        if any(re.search(r'\b' + re.escape(keyword) + r'\b', sentence.lower()) for keyword in keywords):
            filtered_sentences.append(sentence)

    return ' '.join(filtered_sentences)  # Return the filtered sentences as a single string

# Function to send the filtered sentences to the ChatGPT API
def process_with_chatgpt(filtered_text):
    try:
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "user", "content": f"Here's the ESG information to process:\n{filtered_text}"}
            ],
            max_tokens=300
        )
        
        # Get the response content
        # response_text = response.choices[0].message['content']
        # return response_text.strip()
        return response
    
    except Exception as e:
        print(f"An error occurred: {e}")
        return None

# Filter ESG-related sentences from the text
filtered_text = filter_esg_sentences(esg_text, esg_keywords)
print("Filtered ESG Sentences:")
print(filtered_text)

# Send the filtered sentences to ChatGPT and get the response
if filtered_text:
    gpt_response = process_with_chatgpt(filtered_text)
    print("\nChatGPT Response:")
    print(gpt_response)

Filtered ESG Sentences:

Our company emitted 2000 tons of CO2 in 2023, which reflects a significant reduction in carbon emissions. The diversity ratio of our workforce is now 45%, showing improvement in inclusivity. We have achieved 70% renewable energy usage this year, a notable increase from last year.

ChatGPT Response:
ChatCompletion(id='chatcmpl-B9Zl0V9xVCuKVX0FXmK5MYmZyEQHJ', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Overall, our ESG performance in 2023 has improved across various metrics. Our carbon emissions have decreased significantly, demonstrating our commitment to reducing our environmental impact. Additionally, our workforce has become more diverse, showcasing our efforts to promote inclusivity and equality within our organization. The increase in renewable energy usage also highlights our dedication to sustainability and reducing our dependency on fossil fuels. Moving forward, we will continue to prioritize ESG i

In [11]:
gpt_response.choices[0].message.content

'Overall, our ESG performance in 2023 has improved across various metrics. Our carbon emissions have decreased significantly, demonstrating our commitment to reducing our environmental impact. Additionally, our workforce has become more diverse, showcasing our efforts to promote inclusivity and equality within our organization. The increase in renewable energy usage also highlights our dedication to sustainability and reducing our dependency on fossil fuels. Moving forward, we will continue to prioritize ESG initiatives and strive for further progress in these areas.'

In [14]:
esg_indicators = {
    'carbon_emissions': "The total amount of carbon emissions in tons.",
    'diversity_ratio': "The percentage of diverse individuals in the workforce.",
    'renewable_energy_usage': "The percentage of energy used that comes from renewable sources.",
    'women_in_leadership': "The percentage of women in leadership positions."
}

response_esg_indicators = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "user", "content": f"Here's the ESG indicator and explanation:\n{esg_indicators}"}
            ],
            max_tokens=300
        )
response_esg_indicators

ChatCompletion(id='chatcmpl-B9ZrIbiJdbKj04stnlj2NafSAmfIS', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="ESG Indicator: women_in_leadership\nExplanation: This indicator measures the percentage of women who hold leadership positions within a company. Companies with a higher percentage of women in leadership positions are often seen as more inclusive, diverse, and better at fostering gender equality in the workplace. This indicator can also indicate a company's commitment to promoting gender diversity and equality within the organization.", refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None))], created=1741622492, model='gpt-3.5-turbo-0125', object='chat.completion', service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=76, prompt_tokens=83, total_tokens=159, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens

In [15]:
response_esg_summary = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "user", "content": f"Give me the ESG information in json format for this report"}
            ],
            max_tokens=300
        )
response_esg_summary

ChatCompletion(id='chatcmpl-B9ZsaQnctH83nF5aEqCJXk7t9jWzo', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='{\n  "company": "XYZ Corporation",\n  "year": 2021,\n  "ESG": {\n    "environmental": {\n      "carbon_emissions": "1000 metric tons",\n      "energy_consumption": "5000 MWh",\n      "water_usage": "1 million gallons"\n    },\n    "social": {\n      "diversity_ratio": "50% female employees",\n      "employee_satisfaction": "80%",\n      "community_investment": "$100,000"\n    },\n    "governance": {\n      "board_diversity": "30% female directors",\n      "executive_compensation": "$1 million",\n      "corporate_governance_score": "90"\n    }\n  }\n}', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None))], created=1741622572, model='gpt-3.5-turbo-0125', object='chat.completion', service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=154, prompt_tokens=19, total_toke

In [16]:
response_esg_summary.choices[0].message.content

'{\n  "company": "XYZ Corporation",\n  "year": 2021,\n  "ESG": {\n    "environmental": {\n      "carbon_emissions": "1000 metric tons",\n      "energy_consumption": "5000 MWh",\n      "water_usage": "1 million gallons"\n    },\n    "social": {\n      "diversity_ratio": "50% female employees",\n      "employee_satisfaction": "80%",\n      "community_investment": "$100,000"\n    },\n    "governance": {\n      "board_diversity": "30% female directors",\n      "executive_compensation": "$1 million",\n      "corporate_governance_score": "90"\n    }\n  }\n}'

In [22]:
response_esg_allin = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "user", "content": f"Give me the ESG information in json format from below:\n{filtered_text}"}
            ],
            max_tokens=300
        )
response_esg_allin

ChatCompletion(id='chatcmpl-B9aSRUO0vO4MH6Zl6RwaIVlYgrPf8', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='{\n  "CO2_emissions": {\n    "year": 2023,\n    "amount_tons": 2000,\n    "reduction_percentage": "significant"\n  },\n  "workforce_diversity": {\n    "year": 2023,\n    "ratio": "45%"\n  },\n  "renewable_energy_usage": {\n    "year": 2023,\n    "percentage": "70%"\n  }\n}', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None))], created=1741624795, model='gpt-3.5-turbo-0125', object='chat.completion', service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=92, prompt_tokens=79, total_tokens=171, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

In [23]:
response_esg_allin.choices[0].message.content

'{\n  "CO2_emissions": {\n    "year": 2023,\n    "amount_tons": 2000,\n    "reduction_percentage": "significant"\n  },\n  "workforce_diversity": {\n    "year": 2023,\n    "ratio": "45%"\n  },\n  "renewable_energy_usage": {\n    "year": 2023,\n    "percentage": "70%"\n  }\n}'

In [21]:
response_esg_intext = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "user", "content": f"i would like to receive ESG information in json format"}
            ],
            max_tokens=300
        )
response_esg_intext.choices[0].message.content

"I'm sorry, but I do not have the capability to provide ESG information in JSON format. However, you can find ESG data in JSON format on various financial data providers or ESG research websites. Some companies also provide API access to their ESG data which can be fetched in JSON format."